# 17: Write your first compiler extension

**Level:** Advanced developer  
**Before you start:** Notebook 12; Python classes.  
**Resources:** CPU unless an optional remote step is enabled.

Implement a small real transformation, plug it into FlagQuantum, and prove it preserves the result.

Run each cell in order. All core calculations are written in this notebook.

## 1. Define a narrowly supported rewrite

Our compiler removes adjacent, unannotated X gates acting on the same qubit. It leaves every other instruction untouched. The manifest declares what the extension is; negotiation rejects capabilities it does not provide.

In [ ]:
from dataclasses import replace
import flagquantum as fq
import torch
from flagquantum.ecosystem.extensions import (
    ExtensionManifest,
    CapabilityResponse,
    extension_scope,
    run_compiler_conformance,
)


class CancelAdjacentX:
    manifest = ExtensionManifest(
        name="workshop_x",
        version="0.1.0",
        kind="compiler",
        capabilities=frozenset({"circuit_ir"}),
    )

    def __init__(self):
        self.active = False

    def negotiate(self, request):
        missing = request.required - self.manifest.capabilities
        blockers = []
        if missing:
            blockers.append("Unsupported capabilities: " + str(sorted(missing)))
        if request.require_gradients:
            blockers.append("Gradient support has not been validated")
        if request.dtype is not None or request.device_type is not None:
            blockers.append("Device and dtype negotiation is not supported")
        return CapabilityResponse(
            not blockers, self.manifest.capabilities, tuple(blockers)
        )

    def start(self, config):
        self.active = True

    def close(self):
        self.active = False

    def compile(self, program, *, target=None):
        if not self.active:
            raise RuntimeError("Compiler is not active")
        if target and set(target) - {"basis_gates"}:
            raise ValueError("Only a basis_gates constraint is supported")
        kept = []
        for instruction in program.instructions:
            plain_x = (
                instruction.name == "x"
                and not instruction.params
                and not instruction.metadata
            )
            if (
                kept
                and plain_x
                and kept[-1].name == "x"
                and kept[-1].wires == instruction.wires
                and not kept[-1].params
                and not kept[-1].metadata
            ):
                kept.pop()
            else:
                kept.append(instruction)
        if target and "basis_gates" in target:
            if any(i.name not in target["basis_gates"] for i in kept):
                raise ValueError(
                    "Remaining instructions are outside the requested basis"
                )
        return replace(program, instructions=tuple(kept))


## 2. Exercise the extension through the SDK

The scoped registry lets us test the extension without installing a package. We negotiate its capabilities, start it, invoke the transformation, and close it even if an error occurs.
The named `fq.compile(..., compiler="workshop_x")` path discovers installed entry points; packaging is the final challenge below.

In [ ]:
from flagquantum.core.ir import ensure_circuit_ir
from flagquantum.ecosystem.extensions import CapabilityRequest, ExtensionConfig

q = fq.Circuit(2).x(0).x(0).h(0).cx(0, 1)
extension = CancelAdjacentX()
with extension_scope([extension]) as registry:
    handle = registry.negotiate(
        "compiler", "workshop_x", CapabilityRequest(required=frozenset({"circuit_ir"}))
    )
    handle.start(ExtensionConfig())
    try:
        compiled = handle.invoke("compile", ensure_circuit_ir(q))
    finally:
        handle.close()
print([(i.name, i.wires) for i in compiled.instructions])
assert len(compiled.instructions) == 2
assert not extension.active
torch.testing.assert_close(
    fq.run(q).to_statevector(), fq.run(compiled).to_statevector(), atol=1e-6, rtol=1e-6
)


## 3. Run the SDK checks

These check the interface and lifecycle. Your numerical test above remains necessary: passing the interface checks alone does not establish the correctness of every rewrite.

In [ ]:
report = run_compiler_conformance(CancelAdjacentX())
print(report)


## Make it yours

Add tests for X gates on different wires and for an intervening gate: neither pair should cancel. Then package the extension with a factory and a `compiler.workshop_x` entry under `flagquantum.extensions` in pyproject.toml, following the independently installed QSteed example. Do not advertise hardware compilation or gradient capabilities before testing them.